# The Climate is Changing: How is Our Sentiment?

## Exploring Changing Sentiment in Climate Articles: 2013 to 2023

### Imports and Setup

In [ ]:
# General mports
import os
import sys
import pandas as pd
from collections import Counter
import re

# NLTK
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('stopwords')
#print(stopwords.words('english'))

# XML data
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET
import lxml

# Plotting
import plotly.express as px
import plotly.io as pio
import matplotlib.pyplot as plt

# Gensim and TSNE
from gensim.corpora import Dictionary
from gensim.models import word2vec

# SKLearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.manifold import TSNE as tsne

# ----File Stitching----
# If not in notebooks folder, cd into it
if os.path.basename(os.getcwd()) != "notebooks":
    os.chdir('./notebooks/')
# If a file is in /sources/, access it by telling the system to look at that path as well as current path
sys.path.append(os.path.join(os.getcwd(), '..', 'sources'))
# A check
os.getcwd()

In [ ]:
# General imports
import numpy as np 
import pandas as pd 
import os
from collections import Counter
import re


In [ ]:
%run data_extraction.ipynb

In [ ]:
%run initial_exploration.iypnb

### Combine then Split TOKEN Table by Year

In [ ]:
# Import from data extraction notebook
TOKEN

In [ ]:
# Replace TOKEN.index = TOKEN.index.set_levels([2013, 2013, 2013, 2018, 2018, 2018, 2023, 2023, 2023], level=0) with below (Claude)
mapping = {0:2013, 1:2013, 2:2013, 3:2018, 4:2018, 5:2018, 6:2023, 7:2023, 8:2023}
TOKEN.index = TOKEN.index.map(lambda x: (mapping[x[0]], x[1]))
TOKEN

In [ ]:
TOKEN_2013 = TOKEN[TOKEN.index, level = 0 == '2013']
TOKEN_2018 = TOKEN[TOKEN.index == '2018']
TOKEN_2023 = TOKEN[TOKEN.index == '2023']

TOKENS = [TOKEN_2013, TOKEN_2018, TOKEN_2023]

# Drop na
for token in TOKENS:
    token = token.dropna()

In [ ]:
TOKEN_2013

### Plot Top Vocab Words

In [ ]:
def plot_top_vocab_words(token):
    # Join tokens into one string per doc_id (Claude)
    docs = token.groupby(level=0)['term_str'].apply(' '.join)

    # Create engine, model
    engine = CountVectorizer()
    model = engine.fit_transform(token['term_str']) # Claude suggested this, passes as series so it's accepted by CountVectorizer()

    # Extract count matrix
    ct = engine.get_feature_names_out()

    # Data frame
    X = pd.DataFrame(model.toarray(), columns=ct)
    X.index.name = 'doc_id'
    
    return X

for token in TOKENS:
    plot_top_vocab_words(token)

In [ ]:
X.T

In [ ]:
# Quick look at length of documents with stopwords
len = X.T.sum().to_frame('n')
# Show stats
len.sort_values('n').plot.barh(figsize=(10,10))
plt.show()

In [ ]:
# Quick look at documents without stopwords
stop_words = set(stopwords.words('english'))
word_tokens = word_tokenize(X)


Observation: Articles get longer over time, at least in this sample of documents.

In [ ]:
X

### Explore Word Distribution Over Time

In [ ]:
def plot_words(words):
    window = int(n_chunks / 10)
    ax = CHUNK[words.split(",")].rolling(window).mean().plot(kind='line', figsize=(20,5), lw=3)
    ax.axhline(CHUNK['anne'].mean(), ls='--', color='gray', lw=2)
    sns.despine()
    plt.title(words)
    plt.show()